
# ThreadPoolExecutor – kleines Lehr-Notebook

Dieses Notebook zeigt Dir Schritt für Schritt:

1. den Unterschied zwischen `submit(...)`, `map(...)` und `as_completed(...)`
2. wie man eine Worker-Funktion mit **mehreren Parametern** aufruft
3. wie man **eine Liste von Pfaden** verarbeitet und dabei zusätzliche feste Parameter wie `limit=10` und `etwas=True` mitgibt

Die Beispiele sind bewusst einfach gehalten und eignen sich gut zum Lernen und Testen in Jupyter / VS Code.


In [ ]:

from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import repeat
from functools import partial
import time
import pandas as pd



## 1) Eine einfache Worker-Funktion

Wir definieren zuerst eine kleine Funktion, die:
- einen `path`
- ein `limit`
- ein `etwas`
- und zusätzlich eine `pause`

entgegennimmt.

Sie wartet kurz und gibt dann ein kleines Ergebnisobjekt zurück.


In [ ]:

def irgendwas(path: str, limit: int, etwas: bool, pause: float = 1.0):
    time.sleep(pause)
    return {
        "path": path,
        "limit": limit,
        "etwas": etwas,
        "pause": pause,
        "zeichen_im_path": len(path),
        "min_laenge_und_limit": min(len(path), limit),
    }

paths = [
    "/ifs/finance",
    "/ifs/sales",
    "/ifs/logs",
    "/ifs/projects",
]

print(paths)



## 2) Variante A: `submit(...)` mit mehreren Parametern

Das ist oft die **klarste und flexibelste** Lösung.

Wichtig ist die Schreibweise:

```python
executor.submit(irgendwas, path, 10, True)
```

Also:
- zuerst die Funktion
- danach die Argumente

**Nicht** so:

```python
executor.submit(irgendwas(path, 10, True))
```

Denn das würde die Funktion sofort ausführen.


In [ ]:

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [
        executor.submit(irgendwas, path, 10, True, 1.0)
        for path in paths
    ]

    results = [future.result() for future in futures]

pd.DataFrame(results)



## 3) Variante B: `submit(...)` + `as_completed(...)`

Hier werden die Ergebnisse **in der Reihenfolge der Fertigstellung** verarbeitet.

Das ist besonders nützlich, wenn einzelne Tasks unterschiedlich lange dauern.


In [ ]:

pauses = [3.0, 1.0, 2.0, 0.5]

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [
        executor.submit(irgendwas, path, 10, True, pause)
        for path, pause in zip(paths, pauses)
    ]

    fertig_reihenfolge = []
    for future in as_completed(futures):
        result = future.result()
        fertig_reihenfolge.append(result)
        print(f"Fertig: {result['path']} | Pause={result['pause']}")

pd.DataFrame(fertig_reihenfolge)



### Merke
Mit `as_completed(...)` bekommst Du die Resultate **nicht in der Reihenfolge der Eingabeliste**,  
sondern in der Reihenfolge, in der die Threads wirklich fertig werden.



## 4) Variante C: `map(...)` mit mehreren Parametern

Wenn nur `path` variiert und die anderen Parameter konstant bleiben sollen, ist `map(...)` zusammen mit `repeat(...)` sehr elegant.

Hier bekommt jeder Worker:
- seinen eigenen `path`
- immer `limit = 10`
- immer `etwas = True`
- immer `pause = 1.0`


In [ ]:

with ThreadPoolExecutor(max_workers=4) as executor:
    results = executor.map(
        irgendwas,
        paths,
        repeat(10),
        repeat(True),
        repeat(1.0),
    )

    map_results = list(results)

pd.DataFrame(map_results)



### Wichtig
`map(...)` liefert die Ergebnisse normalerweise **in der Reihenfolge der Eingaben** zurück,  
auch wenn einzelne Tasks intern früher fertig waren.



## 5) Variante D: Wrapper-Funktion

Manchmal ist es angenehm, die festen Zusatzparameter in eine kleine Wrapper-Funktion zu verpacken.


In [ ]:

def worker_wrapper(path: str):
    return irgendwas(path, 10, True, 1.0)

with ThreadPoolExecutor(max_workers=4) as executor:
    wrapped_results = list(executor.map(worker_wrapper, paths))

pd.DataFrame(wrapped_results)



## 6) Variante E: `functools.partial`

Mit `partial(...)` kannst Du bestimmte Parameter fest vorbelegen.  
Das ist sehr praktisch, wenn nur noch `path` offen bleiben soll.


In [ ]:

worker_partial = partial(irgendwas, limit=10, etwas=True, pause=1.0)

with ThreadPoolExecutor(max_workers=4) as executor:
    partial_results = list(executor.map(worker_partial, paths))

pd.DataFrame(partial_results)



## 7) Kleiner Direktvergleich: `as_completed(...)` vs. normale Reihenfolge

Hier sieht man den Unterschied besonders gut.


In [ ]:

def worker_demo(name: str, pause: float):
    time.sleep(pause)
    return f"{name} fertig nach {pause}s"

tasks = [
    ("Task A", 3.0),
    ("Task B", 1.0),
    ("Task C", 2.0),
]


In [ ]:

print("MIT as_completed():")
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(worker_demo, name, pause) for name, pause in tasks]
    for future in as_completed(futures):
        print(future.result())


In [ ]:

print("OHNE as_completed() – normale Future-Reihenfolge:")
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(worker_demo, name, pause) for name, pause in tasks]
    for future in futures:
        print(future.result())



## 8) Genau Dein Anwendungsfall: Liste von Pfaden + feste Zusatzparameter

Angenommen, Dein Worker sieht so aus:

```python
def worker(path: str, limit: int, etwas: bool):
    ...
```

und Du willst eine Liste von Pfaden verarbeiten, wobei **jeder** Worker zusätzlich
- `limit = 10`
- `etwas = True`

bekommen soll.

Dann ist diese Form sehr typisch und sehr gut lesbar:


In [ ]:

def worker(path: str, limit: int, etwas: bool):
    time.sleep(0.5)
    return f"path={path} | limit={limit} | etwas={etwas}"

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [
        executor.submit(worker, path, 10, True)
        for path in paths
    ]

    results = [future.result() for future in futures]

for r in results:
    print(r)



## 9) Alternative für genau denselben Fall mit `map(...)`

Auch das geht sehr gut:


In [ ]:

with ThreadPoolExecutor(max_workers=4) as executor:
    results = executor.map(worker, paths, repeat(10), repeat(True))

for r in results:
    print(r)



## 10) Praktisches Fazit

Für Deinen Fall sind diese zwei Varianten besonders wichtig:

### A) Flexibel und sehr klar
```python
futures = [executor.submit(worker, path, 10, True) for path in paths]
```

### B) Elegant bei konstanten Zusatzparametern
```python
results = executor.map(worker, paths, repeat(10), repeat(True))
```

## Meine Empfehlung
Für Lernzwecke und für den IFS-Kontext ist meistens **`submit(...)`** am klarsten,  
vor allem wenn Du später noch:
- Fehler je Task behandeln,
- Fortschritt loggen,
- oder `as_completed(...)` verwenden willst.

Wenn Du dagegen nur sehr kompakt dieselbe Funktion auf viele Pfade anwenden willst, ist `map(...)` mit `repeat(...)` sehr schön.
